# CS570 Team 14 — GCN Modifiability Prediction
## Colab Setup & Training Notebook

**Before running:** Go to `Runtime > Change runtime type` and select **GPU** (T4 is fine).

### Workflow
1. Check GPU
2. Mount Google Drive
3. Clone / pull repo from GitHub
4. Install dependencies
5. Set up folder structure on Drive
6. Upload Rico data
7. Preprocess
8. Train
9. Run ablation matrix

## 1. Check GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. Mount Google Drive

All processed data and checkpoints are saved here so they persist between Colab sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone / Pull Repo from GitHub

In [ ]:
import os

REPO_URL = "https://github.com/nadiarvi/cs570-aiml"
REPO_DIR = "/content/cs570-aiml"
BRANCH   = "colab"   # use the colab branch

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
# Add repo root to Python path so src.* imports work
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("sys.path[0]:", sys.path[0])

## 4. Install Dependencies

In [ ]:
!pip install -q sentence-transformers scikit-learn seaborn tqdm
# torch, numpy, pandas are already available in Colab

## 5. Set Up Drive Folder Structure

All large files (Rico JSONs, processed graphs, checkpoints, embedding cache) live on Drive so they survive session restarts.

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/cs570-project"

PATHS = {
    "raw":         f"{DRIVE_ROOT}/data/raw",
    "processed":   f"{DRIVE_ROOT}/data/processed",
    "splits":      f"{DRIVE_ROOT}/data/splits",
    "gold":        f"{DRIVE_ROOT}/data/gold",
    "checkpoints": f"{DRIVE_ROOT}/results/checkpoints",
    "logs":        f"{DRIVE_ROOT}/results/logs",
    "figures":     f"{DRIVE_ROOT}/results/figures",
}

for name, path in PATHS.items():
    os.makedirs(path, exist_ok=True)
    print(f"  {name:12s} -> {path}")

EMBEDDING_CACHE = f"{DRIVE_ROOT}/embedding_cache.json"
print(f"\nEmbedding cache: {EMBEDDING_CACHE}")

## 6. Upload Rico Data

Download the Rico view hierarchy JSONs and place them in your Drive.

**Option A — Download directly from Rico (if the server allows it):**
```
Source: http://interactionmining.org/rico
File:   ui_layout_vectors.zip
```

**Option B — Upload manually** from your local machine to `MyDrive/cs570-project/data/raw/`

Expected structure after extraction:
```
MyDrive/cs570-project/data/raw/<app_id>/<screen_id>.json
```

Run the cell below to verify the data is in place before proceeding.

In [ ]:
import glob

json_files = glob.glob(f"{PATHS['raw']}/**/*.json", recursive=True)
print(f"Found {len(json_files)} Rico JSON files")
if json_files:
    print("Sample:", json_files[0])
else:
    print("WARNING: No JSON files found. Upload Rico data to Drive before preprocessing.")

## 7. Preprocess

Converts Rico JSONs → graph `.pt` files saved to Drive.

- `--workers 1` avoids multiprocessing issues in Colab
- Re-running is safe — already-processed files are skipped
- Start with `--max_screens 500` to do a quick sanity check first

In [ ]:
# Quick sanity check: preprocess 500 screens first
!python src/data/preprocess.py \
    --rico_dir         {PATHS['raw']} \
    --out_dir          {PATHS['processed']} \
    --split_path       {PATHS['splits']}/split_seed42.json \
    --label_mode       contextual \
    --embedding_cache_path {EMBEDDING_CACHE} \
    --workers          1 \
    --max_screens      500

In [ ]:
# Full preprocessing (5K–10K screens recommended before full 72K)
# Remove --max_screens to process everything
!python src/data/preprocess.py \
    --rico_dir         {PATHS['raw']} \
    --out_dir          {PATHS['processed']} \
    --split_path       {PATHS['splits']}/split_seed42.json \
    --label_mode       contextual \
    --embedding_cache_path {EMBEDDING_CACHE} \
    --workers          1 \
    --max_screens      10000

In [ ]:
# Also preprocess with local_only mode (needed for circularity safeguard)
!python src/data/preprocess.py \
    --rico_dir         {PATHS['raw']} \
    --out_dir          {PATHS['processed']} \
    --split_path       {PATHS['splits']}/split_seed42.json \
    --label_mode       local_only \
    --embedding_cache_path {EMBEDDING_CACHE} \
    --workers          1 \
    --max_screens      10000

In [ ]:
# Verify processed files exist
train_pts = glob.glob(f"{PATHS['processed']}/train/**/*.pt", recursive=True)
val_pts   = glob.glob(f"{PATHS['processed']}/val/**/*.pt",   recursive=True)
print(f"Train graphs: {len(train_pts)}")
print(f"Val graphs:   {len(val_pts)}")

## 8. Train a Single Model

Trains on heuristic labels, early-stops on heuristic val Macro-F1.
Checkpoint and metadata are saved to Drive.

In [ ]:
import json
from src.train import train

with open(f"{REPO_DIR}/experiments/configs/colab_gcn_2l_all_contextual.json") as f:
    config = json.load(f)

# Override paths in case Drive root differs
config["processed_dir"] = PATHS["processed"]
config["save_dir"]       = f"{PATHS['checkpoints']}/gcn_2l_all_contextual"
config["save_root"]      = PATHS["checkpoints"]

metadata = train(config)
print(f"\nBest val Macro-F1: {metadata['best_val_macro_f1']:.4f} (epoch {metadata['best_epoch']})")
print(f"Checkpoint: {metadata['checkpoint_path']}")

## 9. Run Full Ablation Matrix

Trains all 9 configs and evaluates each on gold labels.
Results are appended to `ablation_results.csv` on Drive.

> **Note:** Requires gold labels at `data/gold/gold_test_labels.csv` on Drive.

In [ ]:
from src.ablation import run_ablation

with open(f"{REPO_DIR}/experiments/configs/colab_ablation_base.json") as f:
    base_config = json.load(f)

# Override paths
base_config["processed_dir"]    = PATHS["processed"]
base_config["gold_labels_path"] = f"{PATHS['gold']}/gold_test_labels.csv"
base_config["save_root"]        = PATHS["checkpoints"]

out_csv = f"{DRIVE_ROOT}/results/ablation_results.csv"
run_ablation(base_config, out_csv=out_csv)

In [ ]:
# View results table
import pandas as pd

results = pd.read_csv(out_csv)
cols = ["name", "model", "label_mode", "heuristic_val_macro_f1", "gold_macro_f1", "gold_accuracy"]
print(results[cols].sort_values("gold_macro_f1", ascending=False).to_string(index=False))

## Utilities

In [ ]:
# Check Drive disk usage
!du -sh {DRIVE_ROOT}/*

In [ ]:
# Monitor GPU memory during training
!nvidia-smi --query-gpu=memory.used,memory.free,utilization.gpu --format=csv

In [ ]:
# Push results back to GitHub from Colab
import subprocess

# Copy ablation CSV into the repo before committing
!cp {out_csv} {REPO_DIR}/results/ablation_results.csv

!git -C {REPO_DIR} config user.email "nadia.arvi@gmail.com"
!git -C {REPO_DIR} config user.name "nadiarvi"
!git -C {REPO_DIR} add results/ablation_results.csv
!git -C {REPO_DIR} commit -m "Add ablation results from Colab run"
!git -C {REPO_DIR} push origin colab